Buscar criar um histórico de linhas para cada veículo, ver se segue algum padrão ou é totalmente randômico.

In [1]:
import pandas as pd

In [2]:
veiculos = pd.read_csv('dados_geral/veiculo/veiculos_em_linha.csv')
registros_viagem = pd.read_csv('dados_geral/registros_viagem/viagem_informada.csv')

In [3]:
print(veiculos.info())
print(registros_viagem.info())



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6598 entries, 0 to 6597
Data columns (total 29 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   data                            6598 non-null   object 
 1   modo                            6598 non-null   object 
 2   id_veiculo                      6598 non-null   object 
 3   ano_fabricacao                  6598 non-null   int64  
 4   carroceria                      6598 non-null   object 
 5   data_ultima_vistoria            5854 non-null   object 
 6   id_carroceria                   6598 non-null   int64  
 7   id_chassi                       6598 non-null   int64  
 8   id_fabricante_chassi            6598 non-null   int64  
 9   id_interno_carroceria           6598 non-null   int64  
 10  id_planta                       6598 non-null   int64  
 11  indicador_ar_condicionado       6598 non-null   bool   
 12  indicador_elevador              65

Tratando os dados dos registros: 

Buscando e lidando com valores nulos no route_id, no id_veiculo e no shape.

In [4]:
linhas_route_na = registros_viagem["route_id"].isna()
#print(registros_viagem.loc[linhas_route_na,"id_veiculo"].isna())
linhas_veiculo_na = registros_viagem["id_veiculo"].isna()
#print(registros_viagem.loc[linhas_veiculo_na,"route_id"].isna().sum())
#print(linhas_route_na.sum())
#print(linhas_veiculo_na.sum())
#print(registros_viagem.loc[linhas_veiculo_na,:].head())
#print(registros_viagem.loc[linhas_route_na,:].head())

#Vendo se é possível completar os route_id com o "serviço", número da linha
#Serviço dos que não tem rota:
servicos = registros_viagem.loc[linhas_route_na,"servico"]
#Dos que tem rota nula, quantos tem serviço nulo?
print("Serviços \"nulos\" nas rotas nulas:",servicos.isna().sum())
#O resultado foi zero, então seria de grande valor tentar completar os route_id com o serviço
#Para isso, devo consultar uma outra tabela. Por enquanto, vamos jogar esses valores para fora e trabalhar com a tabela para ver o resultado.

#registros_viagem = registros_viagem.loc[~linhas_route_na,:]

registros_viagem = registros_viagem.loc[~linhas_veiculo_na,:]#Esse aqui realmente não tem o que fazer
print("Rotas \"nulas\":",linhas_route_na.sum())
print("Veiculos \"nulos\":",linhas_veiculo_na.sum())


Serviços "nulos" nas rotas nulas: 0
Rotas "nulas": 1003
Veiculos "nulos": 289


Lidando com registros com ids de rotas vazios: GFTS-> Routes tem a relação ROUTEID e ROUTE SHORT NAME, que aparentemente é o serviço, que vimos que é sempre preenchido no registro. Verificar se de fato é o mesmo serviço, e completar se for o caso

In [5]:
rotas = pd.read_csv(r'dados_geral\gfts\gtfs\routes.csv')

Vendo se o servico e o route_short_name são a mesma coisa

In [6]:
todos_os_servicos = set(rotas["route_short_name"].unique())
servicos_registros = set(registros_viagem["servico"].unique())
servicos_reg_fora_rota = servicos_registros-todos_os_servicos
servicos_reg_intersecao = servicos_registros - servicos_reg_fora_rota
if (servicos_reg_fora_rota) != set():
    #mais preocupante
    print("Existem serviços nos registros de viagem que não estão presentes na tabela de rotas")
    print(len(servicos_reg_fora_rota))
if (todos_os_servicos-servicos_registros) != set():
    #Tudo bem, a rota apenas não foi feita.
    print("Existem serviços na tabela de rotas que não estão presentes nos registros de viagem")
    print(todos_os_servicos-servicos_registros)  
#Dos que estão sem rota, quantos tem serviço que não dá match com a tabela de rotas? 
total_servicos_fora_rota = registros_viagem.loc[linhas_route_na,:]
total_servicos_fora_rota = total_servicos_fora_rota.loc[total_servicos_fora_rota["servico"].isin(servicos_reg_fora_rota),:].shape[0]
print("Apenas", total_servicos_fora_rota, " nulos tem serviços que não estão presentes na tabela de rotas")


Existem serviços nos registros de viagem que não estão presentes na tabela de rotas
11
Existem serviços na tabela de rotas que não estão presentes nos registros de viagem
{'SN817', 'SP852', 'SR350', '68', 'SVA326', '13', 'SN746', 'SE397', 'SN011', '2307', '795', '2804TP', 'SPA232', 'SP455', '18', '404', 'SP393', '2334', '2145', '2015', '537', '14', 'SE01', 'SV301', 'SPA554', '813', 'LECD66', '53', 'LECD73', '434', 'SN821', 'LECD125', 'SN386', '991', '41', '582', '886', 'LECD43', 'SR385', '2303TP', 'SVA905', 'SP361', '2342TP', '782', 'SN583', 'SN107', '398', '71', '2803', 'LECD69', 'SN584', '2380', '38', 'SN552', '747', 'SN201', '445', 'SE008', 'SPA397', 'SE100', '2338', 'SV843', 'SV942', 'LECD58', 'LECD71', '609', 'LECD125TP', 'SN778', '914', '428A', '2110TP', '376', '503', 'LECD54', '772', 'ESP02', '2339', 'SE393', 'SR335', '218', '811', 'SN309', 'LECD45', 'SN326', '203', '2308TP', '581', '012', '2024', '816', 'SN806', '876', 'LECD78', '751', '806', 'SPB232', 'SP335', '2614', 'SN688',

Dos 1003 registros sem route_id, apenas dois não tem serviço que dê match com o do mapa das rotas. Esses dois vão ser excluidos, o resto vai ser preenchido com o id_route da tabela GFTS

In [7]:
#verificar duplicatas
print(rotas["route_short_name"].duplicated().sum())

# se houver, apenas usar a primeira ocorrencia para evitar problemas de merge
rotas_unicas = rotas[['route_short_name', 'route_id']].drop_duplicates(subset='route_short_name')

36213


In [9]:
preencher = registros_viagem.loc[linhas_route_na,:]
preencher = preencher.loc[preencher["servico"].isin(servicos_reg_intersecao),:]
#Agora, tenho que fazer o merge com a tabela de rotas para pegar o route_id
print(preencher.route_id.value_counts(dropna=False)) # Para confirmar que é o total
preencher.drop(columns=["route_id"], inplace=True) #tiro a coluna route_id, vai passar a ser a de rotas
indice_original = preencher.index
preencher = preencher.merge(
    rotas_unicas[['route_short_name', 'route_id']], 
    left_on="servico", 
    right_on="route_short_name", 
    how="left",
    suffixes=('', '_novo') # Mantém o original limpo e o vindo do merge com sufixo
)
preencher = preencher.drop(columns=['route_short_name'])
preencher.index = indice_original
#Faz com que as entradas com os índices iguais aos originais sejam atualizadas.
registros_viagem.update(preencher)


route_id
NaN    1001
Name: count, dtype: int64


In [ ]:
print(registros_viagem["route_id"].isna().sum())
print(total_servicos_fora_rota==registros_viagem["route_id"].isna().sum())


2


Se True, preencheu certamente, e basta retirar os valores que sobraram, cujas rotas não permitiam preencher, se False, há problema na lógica.


Agora, buscando os veículos licenciados que tenham aparecido nos registros: são os que temos informações que possibilitam análise.

In [85]:
#ids únicos de veículos e ids unicos de veiculos usados em viagem
ids_em_linha = set(veiculos["id_veiculo"].unique())
ids_registros_viagem = set(registros_viagem["id_veiculo"].unique())

#ids que são veiculos cadastrados mas não aparecem nos registros de viagem, e vice-versa
fora_de_linha_nos_registros = ids_registros_viagem - ids_em_linha # todos que est
em_linha_fora_registros = ids_em_linha - ids_registros_viagem

if fora_de_linha_nos_registros:
    print(f"Há {len(fora_de_linha_nos_registros)} IDs 'fora de linha' presentes nos registros.")
    print(f"Exemplos: {list(fora_de_linha_nos_registros)[:5]}")
    print(len(ids_registros_viagem))
    print(len(ids_em_linha))
if em_linha_fora_registros:
    print(f"Há {len(em_linha_fora_registros)} IDs 'não usados' presentes em linha.")
    print(f"Exemplos: {list(em_linha_fora_registros)[:5]}")
total_fora_de_linha = registros_viagem[registros_viagem["id_veiculo"].isin(fora_de_linha_nos_registros)]

#Prints bobos para ter noção da corretude das funções de conjunto
#print(fora_de_linha_nos_registros.intersection(em_linha_fora_registros)==set())
#print(fora_de_linha_nos_registros.intersection(ids_em_linha)==set())
#print(fora_de_linha_nos_registros.intersection(ids_registros_viagem)==fora_de_linha_nos_registros)
#print(em_linha_fora_registros.intersection(ids_em_linha)==em_linha_fora_registros)


Há 489 IDs 'fora de linha' presentes nos registros.
Exemplos: ['D33188', 'D33107', 'D33330', 'B58115-2', 'D33294']
4851
6598
Há 2236 IDs 'não usados' presentes em linha.
Exemplos: ['B10505', 'B25610', 'D86702', 'M902247', 'M902177']


A solução foi simplificada. Estamos usando apenas a interseção para fazer o históricos. Não vamos lidar com esses dados. Além disso, para especializar nossa abordagem na questão do ar condicionado, vale simplificar a tabela de veículos em linha. O resto, da tabela de registros de viagem vai ser mantido.

In [86]:
#"Veiculos" simplificado
veiculos_simplificado = veiculos.loc[:,['id_veiculo','indicador_ar_condicionado']]

In [87]:
#Join das tabelas
veiculos_em_linha_em_viagem = veiculos_simplificado.merge(
    registros_viagem, 
    on='id_veiculo', 
    how='inner',
    suffixes=('_veic', '_viag')
)
print(veiculos_em_linha_em_viagem.info())
print(veiculos_em_linha_em_viagem["route_id"].nunique())
#Teste: print((veiculos_em_linha_em_viagem.shape[0]+entradas_fora_de_linha.shape[0])==registros_viagem.shape[0])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 248866 entries, 0 to 248865
Data columns (total 15 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   id_veiculo                   248866 non-null  object 
 1   indicador_ar_condicionado    248866 non-null  bool   
 2   data                         248866 non-null  object 
 3   datetime_partida             248866 non-null  object 
 4   datetime_chegada             248866 non-null  object 
 5   datetime_processamento       248866 non-null  object 
 6   datetime_captura             248866 non-null  object 
 7   trip_id                      0 non-null       float64
 8   route_id                     248866 non-null  object 
 9   shape_id                     247806 non-null  object 
 10  servico                      248866 non-null  object 
 11  sentido                      248866 non-null  object 
 12  id_viagem                    248866 non-null  object 
 13 

Agora tenho todos os veículos por cada trip realizada. Fazer agora um map id->route. se houver alguma route muito relevante, eu travo id->rota e analiso essa rota e o ar condicionado. se não, excluo novamente

In [88]:
#Par veiculo - rotas, e frequencia que aparece
contagem_rotas = veiculos_em_linha_em_viagem.groupby(['id_veiculo', 'route_id']).size().reset_index(name='frequencia')
#Frquência relativa de cada rota em relação ao total de trips do veículo
contagem_rotas["frequencia_relativa"] = contagem_rotas['frequencia'] / contagem_rotas.groupby('id_veiculo')['frequencia'].transform('sum')
# Filtragem rigorosa: mantém apenas registros com proporção > 0.5
veiculos_dominantes = contagem_rotas[contagem_rotas['frequencia_relativa'] > 0.5]

# Ordenação para análise (opcional)
veiculos_dominantes = veiculos_dominantes.sort_values(by='frequencia_relativa', ascending=False)

print(veiculos_dominantes.shape)
print(contagem_rotas.groupby('id_veiculo').size().shape)

(2167, 4)
(4362,)


Dada a conclusão, associar o veículo à uma rota, ou não, e usar os dados de shape que existem para trabalhar em cima dos dados da onda de calor.
Penso em enriquecer com gráficos para facilitar a visualização, e tomar a decisão direito.

Testes de integridade: se os valores "se somam", buscar nas, ect

In [ ]:
#Se eu agrupei corretamente
print(veiculos_dominantes.shape==contagem_rotas.groupby('id_veiculo').size().shape)


Visualizações dos resultados: agregar com as infos de ar condicionado também.